In [ ]:
!pip install sentence-transformers


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.multioutput import MultiOutputClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier

# =====================
# Load data
# =====================
df = pd.read_excel("/content/Annotated_Data.xlsx")

df["text"] = df["target_tweet"].astype(str) + " " + df["authentic_reply"].astype(str)

label_columns = ["STANCE", "ACTION", "PERSONALNESS", "POLITENESS"]

X = df["text"]
Y = df[label_columns]


In [ ]:
# =====================
# Encode labels
# =====================
encoders = {}
Y_encoded = pd.DataFrame()

for col in label_columns:
    le = LabelEncoder()
    Y_encoded[col] = le.fit_transform(Y[col])
    encoders[col] = le

# =====================
# Train / Val / Test split
# =====================
X_train, X_temp, Y_train, Y_temp = train_test_split(
    X, Y_encoded, test_size=0.30, random_state=42
)

X_val, X_test, Y_val, Y_test = train_test_split(
    X_temp, Y_temp, test_size=0.50, random_state=42
)

In [ ]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("all-MiniLM-L6-v2")

X_train_vec = embedder.encode(X_train.tolist(), show_progress_bar=True)
X_val_vec = embedder.encode(X_val.tolist())
X_test_vec = embedder.encode(X_test.tolist())


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/18 [00:00<?, ?it/s]

In [ ]:
xgb_base = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="multi:softmax",
    eval_metric="mlogloss",
    tree_method="hist"
)

In [ ]:
model = MultiOutputClassifier(xgb_base)

# =====================
# Train
# =====================
model.fit(X_train_vec, Y_train)

MultiOutputClassifier(estimator=XGBClassifier(base_score=None, booster=None,
                                              callbacks=None,
                                              colsample_bylevel=None,
                                              colsample_bynode=None,
                                              colsample_bytree=0.8, device=None,
                                              early_stopping_rounds=None,
                                              enable_categorical=False,
                                              eval_metric='mlogloss',
                                              feature_types=None,
                                              feature_weights=None, gamma=None,
                                              grow_policy=None,
                                              importance_type=None,
                                              interaction_constraints=None,
                                              learning_rate=0.1, max_bin=None,
                                              max_cat_threshold=None,
                                              max_cat_to_onehot=None,
                                              max_delta_step=None, max_depth=6,
                                              max_leaves=None,
                                              min_child_weight=None,
                                              missing=nan,
                                              monotone_constraints=None,
                                              multi_strategy=None,
                                              n_estimators=300, n_jobs=None,
                                              num_parallel_tree=None, ...))

In [ ]:
preds = model.predict(X_test_vec)

In [ ]:
print("\nTest accuracy per label:")
for i, col in enumerate(label_columns):
    acc = accuracy_score(Y_test.iloc[:, i], preds[:, i])
    print(f"{col}: {acc:.4f}")


Test accuracy per label:
STANCE: 0.5000
ACTION: 0.7083
PERSONALNESS: 0.6000
POLITENESS: 0.5250


In [ ]:
import os
os.makedirs("models", exist_ok=True)

for i, est in enumerate(model.estimators_):
    est.save_model(f"models/xgb_output_{i}.json")
